In [ ]:
'''
TEST VERİLERİ
total_tests / new_tests	Toplam/günlük yapılan test sayısı
positive_rate	pozitif testlerin tüm testlere oranı
(yüksekse muhtemelen yeterince test yapılmıyor demektir)
tests_per_case	Bir vaka bulmak için kaç test yapıldı
tests_units	Test sayımının birimi (kişi mi, test sayısı mı — bazı ülkeler farklı sayıyor)
'''

'\nTEST VERİLERİ\ntotal_tests / new_tests\tToplam/günlük yapılan test sayısı\npositive_rate\tTestlerin yüzde kaçı pozitif çıktı\n(yüksekse muhtemelen yeterince test yapılmıyor demektir)\ntests_per_case\tBir vaka bulmak için kaç test yapıldı\ntests_units\tTest sayımının birimi (kişi mi, test sayısı mı — bazı ülkeler farklı sayıyor)\n'

In [1]:
#temizlenmiş veri setini yüklüyoruz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_countries = pd.read_csv(
    r'C:\Users\nazlican\OneDrive\Masaüstü\CoronaVirusProject\data\processed\countries_clean.csv',
    parse_dates=['date']
)

print('Satır-sütun sayısı:', df_countries.shape)
print('Ülke sayısı:', df_countries['location'].nunique())
print('İlk tarih:', df_countries['date'].min())
print('Son tarih:', df_countries['date'].max())

print(
    'Tekrarlayan ülke-tarih kaydı:',
    df_countries.duplicated(
        subset=['location', 'date']
    ).sum()
)

Satır-sütun sayısı: (324819, 67)
Ülke sayısı: 194
İlk tarih: 2020-01-01 00:00:00
Son tarih: 2024-08-14 00:00:00
Tekrarlayan ülke-tarih kaydı: 0


In [2]:
#ttest sütunlarını ve veri tiplerini kontrol ediyoruz
testing_columns = [
    'location',
    'date',
    'total_tests',
    'new_tests',
    'total_tests_per_thousand',
    'new_tests_per_thousand',
    'new_tests_smoothed',
    'new_tests_smoothed_per_thousand',
    'positive_rate',
    'tests_per_case',
    'tests_units'
]

print(df_countries[testing_columns].info())

display(df_countries[testing_columns].head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324819 entries, 0 to 324818
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   location                         324819 non-null  object        
 1   date                             324819 non-null  datetime64[ns]
 2   total_tests                      75139 non-null   float64       
 3   new_tests                        70747 non-null   float64       
 4   total_tests_per_thousand         75139 non-null   float64       
 5   new_tests_per_thousand           70747 non-null   float64       
 6   new_tests_smoothed               97794 non-null   float64       
 7   new_tests_smoothed_per_thousand  97794 non-null   float64       
 8   positive_rate                    90826 non-null   float64       
 9   tests_per_case                   89386 non-null   float64       
 10  tests_units                      100342 non-

,location,date,total_tests,new_tests,total_tests_per_thousand,new_tests_per_thousand,new_tests_smoothed,new_tests_smoothed_per_thousand,positive_rate,tests_per_case,tests_units
0,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,2020-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,2020-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,2020-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,2020-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
testing_coverage = (
    df_countries
    .groupby('location')
    .agg(
        toplam_satir=('date', 'size'),
        dolu_toplam_test_kaydi=('total_tests', 'count'),
        dolu_yeni_test_kaydi=('new_tests', 'count'),
        dolu_duzeltilmis_test_kaydi=(
            'new_tests_smoothed',
            'count'
        ),
        dolu_pozitiflik_kaydi=('positive_rate', 'count'),
        dolu_test_vaka_orani_kaydi=(
            'tests_per_case',
            'count'
        ),
        dolu_test_birimi_kaydi=('tests_units', 'count')
    )
)

testing_coverage['bos_yeni_test_kaydi'] = (
    testing_coverage['toplam_satir']
    - testing_coverage['dolu_yeni_test_kaydi']
)

testing_coverage['bos_pozitiflik_kaydi'] = (
    testing_coverage['toplam_satir']
    - testing_coverage['dolu_pozitiflik_kaydi']
)

testing_country_counts = pd.Series({
    'Toplam test verisi bulunan ülke': (
        testing_coverage['dolu_toplam_test_kaydi'].gt(0).sum()
    ),
    'Yeni test verisi bulunan ülke': (
        testing_coverage['dolu_yeni_test_kaydi'].gt(0).sum()
    ),
    'Düzeltilmiş test verisi bulunan ülke': (
        testing_coverage['dolu_duzeltilmis_test_kaydi'].gt(0).sum()
    ),
    'Pozitiflik oranı bulunan ülke': (
        testing_coverage['dolu_pozitiflik_kaydi'].gt(0).sum()
    ),
    'Test/vaka oranı bulunan ülke': (
        testing_coverage['dolu_test_vaka_orani_kaydi'].gt(0).sum()
    ),
    'Test birimi bilgisi bulunan ülke': (
        testing_coverage['dolu_test_birimi_kaydi'].gt(0).sum()
    )
})

print(testing_country_counts)

print(
    testing_coverage
    .sort_values('dolu_pozitiflik_kaydi', ascending=False)
    .head(20)
)

Toplam test verisi bulunan ülke         172
Yeni test verisi bulunan ülke           142
Düzeltilmiş test verisi bulunan ülke    169
Pozitiflik oranı bulunan ülke           162
Test/vaka oranı bulunan ülke            161
Test birimi bilgisi bulunan ülke        176
dtype: int64
                      toplam_satir  dolu_toplam_test_kaydi  \
location                                                     
Mexico                        1678                     900   
Malaysia                      1684                     878   
United Arab Emirates          1674                     846   
Argentina                     1678                     870   
Japan                         1674                     866   
Thailand                      1675                     882   
Israel                        1674                     854   
South Korea                   1674                     859   
Canada                        1674                     834   
Slovenia                      1674       

In [5]:
#hangi ülke teslerin nasıl değerlendirmiş.Test sayısına göre mi yapılan kişi sayına göre mi belirliyoruz.
test_unit_by_country = (
    df_countries.loc[
        df_countries['tests_units'].notna(),
        ['location', 'date', 'tests_units']
    ]
    .groupby('location')
    .agg(
        ilk_birim_tarihi=('date', 'min'),
        son_birim_tarihi=('date', 'max'),
        dolu_birim_kaydi=('tests_units', 'count'),
        farkli_test_birimi=('tests_units', 'nunique'),
        test_birimleri=(
            'tests_units',
            lambda values: ', '.join(
                sorted(values.dropna().unique())
            )
        )
    )
)

test_unit_summary = (
    df_countries.loc[
        df_countries['tests_units'].notna(),
        ['location', 'tests_units']
    ]
    .drop_duplicates()
    .groupby('tests_units')['location']
    .nunique()
    .sort_values(ascending=False)
)

print('Test birimine göre ülke sayısı:')
print(test_unit_summary)

print('\nBirden fazla test birimi görülen ülkeler:')
print(
    test_unit_by_country.loc[
        test_unit_by_country['farkli_test_birimi'] > 1
    ]
)

Test birimine göre ülke sayısı:
tests_units
tests performed    138
people tested       23
samples tested      14
units unclear        1
Name: location, dtype: int64

Birden fazla test birimi görülen ülkeler:
Empty DataFrame
Columns: [ilk_birim_tarihi, son_birim_tarihi, dolu_birim_kaydi, farkli_test_birimi, test_birimleri]
Index: []


In [6]:
#total_test nasıl değişmiş zaman içinde bunu kontrol ediyoruz
total_test_changes = (
    df_countries.loc[
        df_countries['total_tests'].notna(),
        ['location', 'date', 'total_tests']
    ]
    .sort_values(['location', 'date'])
    .copy()
)

total_test_changes['degisim'] = (
    total_test_changes
    .groupby('location')['total_tests']
    .diff()
)

total_test_decreases = total_test_changes.loc[
    total_test_changes['degisim'] < 0
]

print(
    'Toplam test sayısının azaldığı kayıt sayısı:',
    len(total_test_decreases)
)

print(total_test_decreases.to_string(index=False))

Toplam test sayısının azaldığı kayıt sayısı: 0
Empty DataFrame
Columns: [location, date, total_tests, degisim]
Index: []


In [7]:
#total_tests kayıtlarının veri setinde nerede olduğunu incelliyoruz
partial_total_test_missing = testing_coverage.loc[
    testing_coverage['dolu_toplam_test_kaydi'].gt(0)
    & testing_coverage['dolu_toplam_test_kaydi'].lt(
        testing_coverage['toplam_satir']
    )
].copy()

missing_total_test_rows = df_countries.loc[
    df_countries['location'].isin(
        partial_total_test_missing.index
    )
    & df_countries['total_tests'].isna(),
    ['location', 'date']
].copy()

valid_total_test_range = (
    df_countries.loc[
        df_countries['total_tests'].notna()
    ]
    .groupby('location')
    .agg(
        ilk_dolu_test_tarihi=('date', 'min'),
        son_dolu_test_tarihi=('date', 'max')
    )
)

missing_total_test_positions = missing_total_test_rows.join(
    valid_total_test_range,
    on='location'
)

missing_total_test_positions['bosluk_konumu'] = 'arada'

missing_total_test_positions.loc[
    missing_total_test_positions['date']
    < missing_total_test_positions['ilk_dolu_test_tarihi'],
    'bosluk_konumu'
] = 'başta'

missing_total_test_positions.loc[
    missing_total_test_positions['date']
    > missing_total_test_positions['son_dolu_test_tarihi'],
    'bosluk_konumu'
] = 'sonda'

total_test_missing_position_summary = (
    missing_total_test_positions
    .groupby(['location', 'bosluk_konumu'])
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=['başta', 'arada', 'sonda'],
        fill_value=0
    )
)

print(total_test_missing_position_summary)

bosluk_konumu  başta  arada  sonda
location                          
Afghanistan      755    141    773
Albania           51    129    774
Algeria          792      0    881
Andorra          253    452    893
Angola           740    110    794
...              ...    ...    ...
Vanuatu          777      6    889
Vietnam          618    102    776
Yemen            783     86    803
Zambia            74     35    781
Zimbabwe         122    104    774

[172 rows x 3 columns]


In [8]:
#negatif günlük test verisi var mı,kontrol ediyoruz.
negative_new_tests = df_countries.loc[
    df_countries['new_tests'].lt(0),
    ['location', 'date', 'new_tests']
].sort_values(['location', 'date'])

print('Negatif yeni test kaydı sayısı:', len(negative_new_tests))
print(negative_new_tests.to_string(index=False))

Negatif yeni test kaydı sayısı: 0
Empty DataFrame
Columns: [location, date, new_tests]
Index: []
